In [5]:
import pandas as pd
df_exp=pd.read_csv("C:/나이스/summary_experience_dataset.csv")
df_new=pd.read_csv("C:/나이스/summary_new_dataset.csv")

df_combined = pd.concat([df_exp['question_text'], df_new['question_text']], ignore_index=True)

# 결과를 새로운 DataFrame으로 만들기
df_questions = pd.DataFrame({'question_text': df_combined})
# 결과 확인
print(df_questions.head())
df_questions.to_csv('questions.csv')

                                       question_text
0                        디자이너로서 앞으로의 목표에 관해서 설명해 주세요
1  협업을 할 때 사교성이 좋은 편인지 궁금합니다 그리고 또 사교성을 키우기 위해 어떤...
2  지원자님이 태어나서 지금까지 한 일들 가운데 가장 후회했던 일이 무엇인가요 한 가지...
3  대학 생활을 보내면서 가장 힘들었던 경험은 무엇인가요 그것을 어떻게 극복하였는지 예...
4                                  가장 자신있는 작업은 무엇일까요


In [3]:
# 질문별 빈도수 계산
question_counts = df_questions['question_text'].value_counts()

# 상위 5개 가장 많이 나온 질문 출력
top_questions = question_counts.head(5)

print("가장 많이 나온 질문 Top 5:")
print(top_questions)


가장 많이 나온 질문 Top 5:
question_text
회사에 입사를 했더니 업무 내용이나 강도가 예상했던 것과 많이 다르다면 어떻게 하시겠습니까                       19
커뮤니케이션을 잘 할 수 있는 지원자님만의 스킬이 있다면 한번 소개해 주실 수 있을까요                         17
회사의 덕목이 무엇이라고 생각하십니까 어떤 회사가 좋은 회사라고 생각하시는지 말씀해 주세요                       16
지금까지 협업을 하면서 가장 어려웠던 것과 그 어려움을 어떻게 해결했는지 함께 말씀해 주시기 바랍니다                 16
전공 외에 본인이 수준급의 실력이 될 정도로 노력한 것이 있다면 그것이 전공과 관련해서 어떤 도움을 주고 있는지도 궁금합니다    15
Name: count, dtype: int64


In [14]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from konlpy.tag import Okt
import re
import random
import numpy as np

# KoNLPy Okt 형태소 분석기 초기화
okt = Okt()

# 텍스트 전처리 함수 (한국어 특화)
def clean_text(text):
    text = re.sub(r"[^가-힣\s]", "", text) # 한글과 공백만 남기고 제거
    return text.strip()

# 텍스트를 형태소 분석하여 명사만 추출하는 함수
def tokenize_korean_text(text):
    return ' '.join(okt.nouns(clean_text(text)))

def group_and_extract_questions(csv_file_path, num_clusters=5):
    """
    CSV 파일에서 질문을 로드하여 유사한 질문끼리 그룹화하고,
    각 그룹에서 랜덤으로 1개씩 질문을 추출합니다.

    Args:
        csv_file_path (str): 질문이 포함된 CSV 파일의 경로.
        num_clusters (int): 질문을 그룹화할 클러스터(그룹)의 개수.

    Returns:
        dict: 각 그룹에서 추출된 질문을 담은 딕셔너리.
              예: {'cluster_0': '질문 내용', 'cluster_1': '다른 질문 내용', ...}
    """
    try:
        df = df_questions
    except FileNotFoundError:
        print(f"오류: '{csv_file_path}' 파일을 찾을 수 없습니다.")
        return {}
    except Exception as e:
        print(f"CSV 파일 읽기 중 오류 발생: {e}")
        return {}

    if 'question_text' not in df.columns:
        print("오류: CSV 파일에 'question_text' 컬럼이 없습니다.")
        return {}

    # 텍스트 전처리 및 토큰화
    df['processed_question'] = df['question_text'].apply(tokenize_korean_text)

    # TF-IDF 벡터화
    # min_df를 조절하여 너무 자주 나오거나 너무 드물게 나오는 단어 필터링 가능
    vectorizer = TfidfVectorizer(max_features=1000, min_df=5, max_df=0.8)
    X = vectorizer.fit_transform(df['processed_question'])

    # K-Means 군집화
    # n_init='auto' 또는 명시적으로 횟수 지정 (예: 10)
    # n_init: KMeans 초기화 시도 횟수. 'auto'는 scikit-learn 버전 1.4부터 기본값이며, 이전 버전에서는 10이 일반적입니다.
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    df['cluster'] = kmeans.fit_predict(X)

    extracted_questions = {}
    for cluster_id in range(num_clusters):
        cluster_questions = df[df['cluster'] == cluster_id]['question_text'].tolist()
        if cluster_questions:
            extracted_questions[f'Cluster_{cluster_id}'] = random.choice(cluster_questions)
        else:
            extracted_questions[f'Cluster_{cluster_id}'] = "해당 클러스터에 질문이 없습니다."
            
    return extracted_questions

if __name__ == "__main__":
    
    
    # 원하는 클러스터(그룹) 개수를 num_clusters로 설정하세요.
    # 이 값에 따라 질문이 몇 개의 그룹으로 나뉠지 결정됩니다.
    # 너무 작으면 광범위하고, 너무 크면 너무 세분화될 수 있습니다.
    num_clusters_to_extract = 7 # 예시: 10개의 그룹으로 나누기

    selected_questions = group_and_extract_questions(df_questions, num_clusters=num_clusters_to_extract)

    print(f"그룹별로 추출된 질문 ({num_clusters_to_extract}개 그룹):")
    for group_name, question in selected_questions.items():
        print(f"- {group_name}: {question}")

    

그룹별로 추출된 질문 (7개 그룹):
- Cluster_0: 남들과 다른 자신만의 자신만의 무기 경쟁력 있다면 그것을 키우기 위해 어떻게 노력하였는지 그리고 자신만의 경쟁력에 대해 설명을 부탁드려 봅니다
- Cluster_1: 지원자님께서는 사내 스터디를 만들어 공부할 수 있는 환경이 된다면 어떤 것을 공부를 할 생각인가요 그 이유도 함께 설명 부탁드립니다
- Cluster_2: 본인만의 스트레스 해소 방법이 있다면 그것은 무엇인가요 그리고 왜 그 방법으로 스트레스를 해소하는지도 같이 말씀해 주세요
- Cluster_3: 지원자분께서 대학에서 배우신 전공 과목들 가운데에서 가장 기억에 남는 과목이 있다면 그 과목을 한 가지만 골라서 저희에게 설명해 주시면 좋을 것 같습니다
- Cluster_4: 이제까지 속해 있었던 조직이나 단체에서 본인은 주로 어떤 역할을 하는 사람이었는지 설명해 주시겠어요
- Cluster_5: 몇 년 전에 있었던 우버 택시와 관련된 논란을 알고 계실 것이라고 생각합니다 면접자님께서는 찬반 양론에 대해 각각 어떤 생각을 가지고 계신가요
- Cluster_6: 일을 하던 중 직무가 적성에 맞지 않는다면 어떻게 하시겠습니까 상황을 가정해 말씀 부탁드립니다
